# Notebook 8 — Análisis de Sentimiento de Noticias (NLP) → MongoDB
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:** Recopilar noticias financieras reales de los 5 tickers del proyecto (o un
dataset simulado realista si no hay noticias disponibles), analizarlas con **VADER**
(Valence Aware Dictionary and sEntiment Reasoner), clasificarlas como
**BULLISH / BEARISH / NEUTRAL**, guardar los resultados en MongoDB, y calcular un
**sentimiento consolidado** por ticker con su respectivo gauge chart.

**Modelo de sentimiento:** VADER (NLTK) — léxico basado en reglas, no requiere
entrenamiento ni GPU, ideal para Colab gratuito.

**Clasificación (sobre el `compound score` de VADER, rango -1 a +1):**
- `compound > 0.05`  → **BULLISH**
- `compound < -0.05` → **BEARISH**
- resto              → **NEUTRAL**

**Prerequisito:** Ninguno de los notebooks anteriores es estrictamente necesario,
pero se conecta a la misma base de datos `spbi` para mantener todo el proyecto unificado.

**Salida:** Colección `noticias` en MongoDB, más un archivo `datos_nlp.json` exportado
para el módulo NLP del frontend.

> Nota: si Yahoo Finance no devuelve noticias reales para algún ticker (frecuente en
> tickers de bolsas pequeñas como VOLCABC1.LM), se usa un dataset simulado pero
> realista de titulares financieros en inglés, para que el módulo nunca se quede vacío.

## Paso 1 — Instalación de librerías

In [ ]:
# Instalación de librerías necesarias
!pip install nltk pymongo[srv] yfinance --quiet
print('✓ Librerías instaladas')

## Paso 2 — Conexión a MongoDB Atlas

In [ ]:
# Conectar a MongoDB Atlas usando el Secret MONGO_URI de Colab
# (Secrets: ícono de llave en el panel izquierdo de Colab)
from google.colab import userdata
from pymongo import MongoClient
from datetime import datetime, timedelta
import json
import random
import re

MONGO_URI = userdata.get('MONGO_URI')
client    = MongoClient(MONGO_URI)
db        = client['spbi']

# Verificar conexión
client.admin.command('ping')
print('✓ Conectado a MongoDB Atlas')
print(f'  Base de datos: spbi')
print(f'  Colecciones disponibles: {db.list_collection_names()}')

## Paso 3 — Semilla fija (para el dataset simulado de respaldo)

In [ ]:
SEED = 42
random.seed(SEED)
print(f'✓ Semilla fija SEED={SEED} aplicada (random)')

## Paso 4 — Descarga del lexicón VADER (NLTK)

In [ ]:
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize

sia = SentimentIntensityAnalyzer()
print('✓ Lexicón VADER descargado y SentimentIntensityAnalyzer listo')

# Prueba rápida
prueba = sia.polarity_scores('Gold prices surge as mining stocks rally on strong earnings')
print(f'  Prueba: {prueba}')

## Paso 5 — Recopilación de noticias

Se intenta obtener noticias REALES vía `yfinance` (el mismo atributo `.news` que ya
usa el ecosistema del proyecto, sin necesitar API keys de pago). Si un ticker no
devuelve noticias (común en tickers de bolsas pequeñas como VOLCABC1.LM), se recurre
a un **dataset simulado pero realista** de titulares financieros en inglés sobre el
sector minero, para que el módulo nunca se quede sin datos que mostrar.

In [ ]:
import yfinance as yf

TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

EMPRESAS = {
    'FSM'         : 'Fortuna Silver Mines',
    'VOLCABC1.LM' : 'Volcan Compañía Minera',
    'ABX.TO'      : 'Barrick Gold',
    'BVN'         : 'Compañía de Minas Buenaventura',
    'BHP'         : 'BHP Group'
}

FUENTES_SIMULADAS = ['Bloomberg', 'CNBC', 'Reuters', 'MarketWatch']

# Banco de titulares simulados realistas (sector minero/metales), en inglés,
# usado SOLO como respaldo cuando yfinance no trae noticias reales para un ticker.
PLANTILLAS_TITULARES = [
    "{empresa} shares rise as gold prices extend rally",
    "{empresa} reports quarterly production above analyst estimates",
    "Silver miners including {empresa} gain on safe-haven demand",
    "{empresa} announces cost-cutting plan amid volatile metal prices",
    "Analysts upgrade {empresa} following strong exploration results",
    "{empresa} stock slides as commodity prices weaken",
    "Mining sector under pressure: {empresa} shares fall on weak demand outlook",
    "{empresa} faces production delays at key mining site",
    "{empresa} maintains steady output despite market headwinds",
    "Investors watch {empresa} closely ahead of earnings report",
    "{empresa} expands operations with new exploration project",
    "Commodity rally lifts {empresa} and peers in mining sector",
    "{empresa} announces dividend amid solid cash flow",
    "Regulatory changes could impact {empresa} operations, analysts say",
    "{empresa} stock steady as market awaits Fed rate decision"
]

def obtener_noticias_yfinance(ticker):
    """
    Intenta obtener noticias reales vía yfinance. Devuelve una lista de dicts
    normalizados, o una lista vacía si no hay noticias disponibles.
    """
    try:
        raw = yf.Ticker(ticker).news or []
    except Exception:
        raw = []

    noticias = []
    for item in raw:
        # yfinance anida los campos reales bajo 'content' en versiones recientes
        contenido = item.get('content', item)
        titulo = contenido.get('title')
        if not titulo:
            continue

        fuente = (contenido.get('provider') or {}).get('displayName', 'Yahoo Finance')
        url    = (contenido.get('canonicalUrl') or {}).get('url', contenido.get('link', ''))
        resumen = contenido.get('summary') or titulo
        fecha_pub = contenido.get('pubDate') or contenido.get('providerPublishTime')

        if isinstance(fecha_pub, (int, float)):
            fecha_pub = datetime.fromtimestamp(fecha_pub).isoformat()
        elif not fecha_pub:
            fecha_pub = datetime.now().isoformat()

        noticias.append({
            'titulo'          : titulo,
            'texto'           : resumen,
            'fuente'          : fuente,
            'url'             : url,
            'fecha_publicacion': fecha_pub,
            'origen'          : 'real'
        })
    return noticias

def generar_noticias_simuladas(ticker, n=6):
    """
    Genera un dataset simulado pero realista de titulares financieros en inglés,
    usado como respaldo cuando yfinance no trae noticias para el ticker.
    """
    empresa = EMPRESAS[ticker]
    plantillas = random.sample(PLANTILLAS_TITULARES, min(n, len(PLANTILLAS_TITULARES)))
    noticias = []
    ahora = datetime.now()
    for i, plantilla in enumerate(plantillas):
        titulo = plantilla.format(empresa=empresa)
        fecha_pub = ahora - timedelta(minutes=random.randint(5, 720))
        noticias.append({
            'titulo'           : titulo,
            'texto'            : titulo + '. Market analysts continue to monitor sector-wide trends affecting mining equities.',
            'fuente'           : random.choice(FUENTES_SIMULADAS),
            'url'              : '',
            'fecha_publicacion': fecha_pub.isoformat(),
            'origen'           : 'simulado'
        })
    return noticias

print('✓ Funciones de recopilación de noticias definidas (yfinance real + respaldo simulado)')

## Paso 6 — Preprocesamiento de texto (limpieza y tokenización)

In [ ]:
def limpiar_texto(texto):
    """
    Limpieza básica de texto para NLP:
    - Quita URLs, caracteres especiales y espacios repetidos.
    - Mantiene puntuación básica (! y ?) porque VADER SÍ le da peso semántico
      (ej. '!!!' intensifica el sentimiento), a diferencia de un preprocesamiento
      clásico de bag-of-words que normalmente la eliminaría.
    """
    texto = re.sub(r'http\S+|www\S+', '', texto)          # quitar URLs
    texto = re.sub(r'[^a-zA-Z0-9\s\.,!?%$-]', '', texto)   # quitar caracteres raros
    texto = re.sub(r'\s+', ' ', texto).strip()             # espacios repetidos
    return texto

def tokenizar(texto):
    """Tokenización simple con NLTK, usada solo para reporte/diagnóstico."""
    return word_tokenize(texto.lower())

# Prueba rápida
ejemplo = "Gold prices SURGE!!! as mining stocks rally... http://example.com/news"
limpio  = limpiar_texto(ejemplo)
print(f'Original : {ejemplo}')
print(f'Limpio   : {limpio}')
print(f'Tokens   : {tokenizar(limpio)}')

## Paso 7 — Análisis de sentimiento con VADER

Se calculan los 4 scores estándar de VADER (`compound`, `pos`, `neg`, `neu`) sobre
el **título** de cada noticia (VADER está optimizado para texto corto tipo social
media/titulares, por lo que analizar el título da resultados más confiables que
analizar párrafos largos de texto financiero formal).

In [ ]:
def analizar_sentimiento(titulo_limpio):
    """
    Analiza el sentimiento de un titular con VADER y lo clasifica en
    BULLISH / BEARISH / NEUTRAL según el compound score.
    """
    scores = sia.polarity_scores(titulo_limpio)
    compound = scores['compound']

    if compound > 0.05:
        etiqueta = 'BULLISH'
    elif compound < -0.05:
        etiqueta = 'BEARISH'
    else:
        etiqueta = 'NEUTRAL'

    return {
        'sentimiento': etiqueta,
        'compound'   : round(compound, 4),
        'pos'        : round(scores['pos'], 4),
        'neg'        : round(scores['neg'], 4),
        'neu'        : round(scores['neu'], 4)
    }

# Prueba rápida
for ejemplo in [
    "Gold prices surge as mining stocks rally on strong earnings",
    "Mining sector under pressure as shares fall on weak demand outlook",
    "Company maintains steady output despite market headwinds"
]:
    r = analizar_sentimiento(limpiar_texto(ejemplo))
    print(f"[{r['sentimiento']:<8}] compound={r['compound']:+.3f} | {ejemplo}")

## Paso 8 — Pipeline completo por ticker: recopilar, limpiar, analizar y guardar

In [ ]:
def procesar_ticker_noticias(ticker):
    """
    Pipeline completo para un ticker:
    1. Intenta obtener noticias reales vía yfinance.
    2. Si no hay (o son muy pocas), completa con noticias simuladas realistas.
    3. Limpia y analiza el sentimiento de cada titular con VADER.
    4. Guarda todas las noticias procesadas en la colección 'noticias' de MongoDB.
    5. Calcula el sentimiento consolidado (promedio de compound) del ticker.
    """
    print(f'\n  [{ticker}] Procesando...')

    noticias_reales = obtener_noticias_yfinance(ticker)
    n_reales = len(noticias_reales)

    MIN_NOTICIAS = 6
    if n_reales < MIN_NOTICIAS:
        faltantes = MIN_NOTICIAS - n_reales
        noticias_simuladas = generar_noticias_simuladas(ticker, n=faltantes)
        noticias = noticias_reales + noticias_simuladas
        print(f'  [{ticker}] {n_reales} noticias reales + {len(noticias_simuladas)} simuladas (respaldo)')
    else:
        noticias = noticias_reales
        print(f'  [{ticker}] {n_reales} noticias reales obtenidas de yfinance')

    documentos = []
    compounds  = []
    for noticia in noticias:
        titulo_limpio = limpiar_texto(noticia['titulo'])
        analisis = analizar_sentimiento(titulo_limpio)
        compounds.append(analisis['compound'])

        documentos.append({
            'ticker'           : ticker,
            'titulo'           : noticia['titulo'],
            'texto'            : noticia['texto'],
            'fuente'           : noticia['fuente'],
            'url'              : noticia['url'],
            'sentimiento'      : analisis['sentimiento'],
            'compound'         : analisis['compound'],
            'pos'              : analisis['pos'],
            'neg'              : analisis['neg'],
            'neu'              : analisis['neu'],
            'fecha_publicacion': noticia['fecha_publicacion'],
            'origen'           : noticia['origen'],
            'created_at'       : datetime.now()
        })

    # ── Guardar en MongoDB (reemplaza noticias previas de este ticker) ──
    db['noticias'].delete_many({'ticker': ticker})
    if documentos:
        db['noticias'].insert_many(documentos)

    # ── Sentimiento consolidado del ticker ───────────────────────────
    compound_promedio = round(sum(compounds) / len(compounds), 4) if compounds else 0.0
    if compound_promedio > 0.05:
        etiqueta_consolidada = 'BULLISH'
    elif compound_promedio < -0.05:
        etiqueta_consolidada = 'BEARISH'
    else:
        etiqueta_consolidada = 'NEUTRAL'

    conteo = {
        'BULLISH' : sum(1 for d in documentos if d['sentimiento'] == 'BULLISH'),
        'BEARISH' : sum(1 for d in documentos if d['sentimiento'] == 'BEARISH'),
        'NEUTRAL' : sum(1 for d in documentos if d['sentimiento'] == 'NEUTRAL'),
    }

    print(f'  [{ticker}] ✓ Sentimiento consolidado: {etiqueta_consolidada} (compound={compound_promedio:+.3f}) | {conteo}')

    return {
        'ticker'              : ticker,
        'n_noticias'          : len(documentos),
        'sentimiento_consolidado': etiqueta_consolidada,
        'compound_promedio'   : compound_promedio,
        'conteo'              : conteo,
        'noticias'            : documentos
    }

print('✓ Función procesar_ticker_noticias definida')

## Paso 9 — Ejecutar pipeline para los 5 tickers

In [ ]:
print('=' * 60)
print('  ANÁLISIS DE SENTIMIENTO NLP (VADER) — 5 TICKERS')
print('=' * 60)

resultados = {}
for ticker in TICKERS:
    try:
        res = procesar_ticker_noticias(ticker)
        resultados[ticker] = res
    except Exception as e:
        print(f'  [{ticker}] ✗ Error inesperado: {e}')

print()
print('=' * 60)
print(f'  ✅ Pipeline completo: {len(resultados)}/{len(TICKERS)} tickers procesados')
print('=' * 60)

## Paso 10 — Verificación en MongoDB

In [ ]:
print('NOTICIAS en MongoDB (resumen por ticker)')
print('-' * 60)
for ticker in TICKERS:
    total = db['noticias'].count_documents({'ticker': ticker})
    bullish = db['noticias'].count_documents({'ticker': ticker, 'sentimiento': 'BULLISH'})
    bearish = db['noticias'].count_documents({'ticker': ticker, 'sentimiento': 'BEARISH'})
    neutral = db['noticias'].count_documents({'ticker': ticker, 'sentimiento': 'NEUTRAL'})
    print(f'  {ticker:<15} total={total:<3} | BULLISH={bullish} BEARISH={bearish} NEUTRAL={neutral}')

print()
print('✅ Colección noticias verificada')

## Paso 11 — Visualización: gauge chart de sentimiento consolidado

Se dibuja un **gauge chart** (indicador tipo velocímetro) por ticker, mostrando el
`compound_promedio` en el rango [-1, +1], con zonas de color BEARISH/NEUTRAL/BULLISH —
la misma idea visual que un semáforo de sentimiento de mercado.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=len(resultados),
    specs=[[{'type': 'indicator'}] * len(resultados)],
    subplot_titles=list(resultados.keys())
)

for i, (ticker, res) in enumerate(resultados.items(), start=1):
    valor = res['compound_promedio']
    fig.add_trace(go.Indicator(
        mode='gauge+number',
        value=valor,
        number={'valueformat': '+.3f'},
        gauge={
            'axis': {'range': [-1, 1]},
            'bar': {'color': 'white'},
            'steps': [
                {'range': [-1, -0.05], 'color': '#EF5350'},   # BEARISH
                {'range': [-0.05, 0.05], 'color': '#FFC107'}, # NEUTRAL
                {'range': [0.05, 1], 'color': '#26A69A'},     # BULLISH
            ],
            'threshold': {
                'line': {'color': 'white', 'width': 3},
                'thickness': 0.8,
                'value': valor
            }
        }
    ), row=1, col=i)

fig.update_layout(
    height=280,
    paper_bgcolor='#0f172a',
    font={'color': '#f8fafc'},
    title_text='Sentimiento Consolidado por Ticker (VADER compound score)',
    margin=dict(l=20, r=20, t=80, b=20)
)
fig.show()

print('✓ Gauge chart generado para los', len(resultados), 'tickers')

## Paso 12 — Exportación a JSON para el frontend (`datos_nlp.json`)

In [ ]:
export_data = {
    'modelo'       : 'VADER (NLTK)',
    'seed'         : SEED,
    'fecha_export' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'umbral_bullish': 0.05,
    'umbral_bearish': -0.05,
    'tickers'      : {}
}

for ticker, res in resultados.items():
    export_data['tickers'][ticker] = {
        'sentimiento_consolidado': res['sentimiento_consolidado'],
        'compound_promedio'      : res['compound_promedio'],
        'conteo'                 : res['conteo'],
        'noticias': [
            {
                'titulo'           : n['titulo'],
                'texto'            : n['texto'],
                'fuente'           : n['fuente'],
                'url'              : n['url'],
                'sentimiento'      : n['sentimiento'],
                'compound'         : n['compound'],
                'fecha_publicacion': n['fecha_publicacion'],
                'origen'           : n['origen']
            }
            for n in res['noticias']
        ]
    }

with open('datos_nlp.json', 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f'✓ Archivo datos_nlp.json exportado ({len(resultados)} tickers)')
print(f'  Tamaño aproximado: {len(json.dumps(export_data)) / 1024:.1f} KB')

# Descargar el archivo desde Colab (opcional)
try:
    from google.colab import files
    files.download('datos_nlp.json')
    print('✓ Descarga iniciada en el navegador')
except Exception as e:
    print(f'  (Descarga automática no disponible en este entorno: {e})')

## Resultado

La colección `noticias` en MongoDB contiene, para cada uno de los 5 tickers:
- Noticias **reales** de Yahoo Finance cuando están disponibles, y noticias
  **simuladas pero realistas** como respaldo cuando no (campo `origen: 'real'/'simulado'`
  para poder distinguirlas siempre).
- Análisis de sentimiento VADER completo (`compound`, `pos`, `neg`, `neu`) por titular.
- Clasificación `BULLISH` / `BEARISH` / `NEUTRAL` según el `compound score`.
- Un **sentimiento consolidado** por ticker (promedio de `compound` de todas sus noticias).

**Gauge chart** generado mostrando visualmente el sentimiento consolidado de los 5 tickers.

**datos_nlp.json** exportado con toda la información, listo para que el frontend construya
su propio feed de noticias sin depender de `Math.random()`.

**Pipeline completo:** `yfinance.news` (+ respaldo simulado) → limpieza → VADER → `noticias` (MongoDB) → gauge chart → `datos_nlp.json` ✓

**Siguiente paso:** agregar un endpoint `/api/noticias` a la API (filtrando por `ticker`,
`fuente` y `sentimiento`, igual que hacen los selectores del módulo NLP del frontend) para
reemplazar `generarNoticias()` por datos reales.